[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C54_DETR_Set_Prediction_Course/04_convergence/04_convergence_family.ipynb)

# 04 · 收敛难题与 DETR 家族演进（可变形注意力 / 双线性梯度 / 匹配翻转 / DN 去噪 query）

目标：把「DETR 为什么要 500 epoch」拆成**两个可度量的根因**，然后亲手实现
Deformable / DN / DINO 各自用来修它们的机制，并用数值证明每一步确实抬高了监督信号密度。

**本 notebook 你会亲手实现：**
1. **双线性插值 + 它对采样坐标的解析梯度**（中心差分校验到 1e-7）——可变形采样的数学地基
2. **多尺度可变形注意力**的完整前向（参考点 + 可学偏移 + softmax 权重 + 跨尺度采样）
3. **计算量账**：全局 attention vs K 点采样，解释「为什么 DETR 做不了多尺度」
4. **根因①**：初始注意力质量 α 的计算与「注意力变尖锐需要多少 epoch」的玩具模型
5. **根因②**：从零写匈牙利算法（暴力对拍），量化**匹配翻转率 φ**
6. **DN 去噪 query 构造**：噪声框生成、标签翻转、**attention mask 隔离**（防 GT 泄漏）
7. **CDN 对比去噪**：正负噪声对的构造与它们与 GT 的 IoU 分布
8. **look forward once vs twice** 的梯度路径差异
9. **一对多监督密度**的账，反推 Group DETR 的收敛加速倍数

> 心智模型：**每一步演进都在抬高同一个量——
> ρ ∝ (每次迭代被监督的 query 数) × (注意力落在目标上的质量) × (匹配未翻转的比例)。**

## 1 · 双线性插值与它对采样坐标的解析梯度

可变形注意力的采样点 `p + Δp` 是**连续坐标**，必须用双线性插值取值才可微。
这一节实现插值、实现解析梯度，并用中心差分校验——
**梯度里最关键的是 `∂v/∂Δp`：它是「学会往哪看」的唯一直接通道。**

In [ ]:
import numpy as np, math, itertools
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

def bilinear_sample(feat, pts):
    '''feat:(H,W,C) 特征图; pts:(N,2) 连续像素坐标 (x,y)。返回 (N,C)。
       越界坐标 clamp 到边界（等价 grid_sample(padding_mode="border")）。'''
    H, W, C = feat.shape
    x = np.clip(pts[:, 0], 0.0, W - 1.0)
    y = np.clip(pts[:, 1], 0.0, H - 1.0)
    x0 = np.clip(np.floor(x).astype(int), 0, W - 2); x1 = x0 + 1
    y0 = np.clip(np.floor(y).astype(int), 0, H - 2); y1 = y0 + 1
    dx = (x - x0)[:, None]; dy = (y - y0)[:, None]
    f00 = feat[y0, x0]; f10 = feat[y0, x1]      # (x0,y0) 与 (x1,y0)
    f01 = feat[y1, x0]; f11 = feat[y1, x1]      # (x0,y1) 与 (x1,y1)
    return (1-dx)*(1-dy)*f00 + dx*(1-dy)*f10 + (1-dx)*dy*f01 + dx*dy*f11

def bilinear_grad_xy(feat, pts):
    '''对采样坐标的解析梯度 (dv/dx, dv/dy)，各 (N,C)。越界处梯度为 0（clamp 让函数变常数）。'''
    H, W, C = feat.shape
    ix = (pts[:, 0] >= 0) & (pts[:, 0] <= W - 1)
    iy = (pts[:, 1] >= 0) & (pts[:, 1] <= H - 1)
    x = np.clip(pts[:, 0], 0.0, W - 1.0); y = np.clip(pts[:, 1], 0.0, H - 1.0)
    x0 = np.clip(np.floor(x).astype(int), 0, W - 2); x1 = x0 + 1
    y0 = np.clip(np.floor(y).astype(int), 0, H - 2); y1 = y0 + 1
    dx = (x - x0)[:, None]; dy = (y - y0)[:, None]
    f00 = feat[y0, x0]; f10 = feat[y0, x1]
    f01 = feat[y1, x0]; f11 = feat[y1, x1]
    # dv/dx = (1-dy)(f10-f00) + dy(f11-f01)   ← **正比于特征图在该处的局部差分**
    gx = ((1-dy)*(f10-f00) + dy*(f11-f01)) * ix[:, None]
    gy = ((1-dx)*(f01-f00) + dx*(f11-f10)) * iy[:, None]
    return gx, gy

H, W, C = 12, 16, 3
feat = rng.normal(size=(H, W, C))
pts = np.stack([rng.uniform(1.2, W-2.2, 40), rng.uniform(1.2, H-2.2, 40)], axis=1)
v = bilinear_sample(feat, pts)
gx, gy = bilinear_grad_xy(feat, pts)

eps = 1e-5
gx_num = (bilinear_sample(feat, pts + [eps, 0]) - bilinear_sample(feat, pts - [eps, 0])) / (2*eps)
gy_num = (bilinear_sample(feat, pts + [0, eps]) - bilinear_sample(feat, pts - [0, eps])) / (2*eps)
print('采样值 shape', v.shape)
print('dv/dx 解析 vs 中心差分  最大误差 %.2e' % np.abs(gx - gx_num).max())
print('dv/dy 解析 vs 中心差分  最大误差 %.2e' % np.abs(gy - gy_num).max())
assert np.allclose(gx, gx_num, atol=1e-7) and np.allclose(gy, gy_num, atol=1e-7)
print()
print('✅ 采样坐标是可微的 —— 这就是「偏移 Δp 可学」的全部理由。')
print('⚠️  推论：特征平坦处 (f10-f00)≈0 -> dv/dΔp≈0 -> 采样点收不到位置梯度，会停在原地。')

In [ ]:
# 梯度回传到特征图：每个采样点只把梯度写到 **4 个格点**
def bilinear_scatter_grad(feat_shape, pts, grad_out):
    '''把上游梯度 (N,C) 按双线性权重散射回特征图 (H,W,C)。'''
    H, W, C = feat_shape
    x = np.clip(pts[:, 0], 0.0, W - 1.0); y = np.clip(pts[:, 1], 0.0, H - 1.0)
    x0 = np.clip(np.floor(x).astype(int), 0, W - 2); x1 = x0 + 1
    y0 = np.clip(np.floor(y).astype(int), 0, H - 2); y1 = y0 + 1
    dx = (x - x0)[:, None]; dy = (y - y0)[:, None]
    g = np.zeros((H, W, C))
    np.add.at(g, (y0, x0), (1-dx)*(1-dy)*grad_out)
    np.add.at(g, (y0, x1), dx*(1-dy)*grad_out)
    np.add.at(g, (y1, x0), (1-dx)*dy*grad_out)
    np.add.at(g, (y1, x1), dx*dy*grad_out)
    return g

go = np.ones((len(pts), C))
g_feat = bilinear_scatter_grad((H, W, C), pts, go)
touched = int((np.abs(g_feat).sum(-1) > 0).sum())
print('采样点数 %d -> 收到梯度的特征格点数 %d / %d 个格点' % (len(pts), touched, H*W))
assert np.allclose(g_feat.sum(), len(pts) * C), '每个采样点的 4 个双线性权重之和恒为 1'
assert touched <= 4 * len(pts)
print()
print('对比两种「梯度分布」：')
print('  全局 attention : HW 个位置各拿 1/HW 的权重 -> 方向由近似均匀的权重决定，**极弱且无指向**')
print('  可变形采样     : 4K 个位置拿到集中的梯度，**而且采样位置本身也收梯度**')
print('✅ 后者多出来的那条 dv/dΔp 通道，是 500 epoch -> 50 epoch 的机制层解释。')

## 2 · 多尺度可变形注意力：完整前向

`MSDeformAttn(z_q, p̂_q, {x_l}) = Σ_l Σ_k A_lqk · x_l(φ_l(p̂_q) + Δp_lqk)`

三个要点：① 参考点 `p̂_q` 归一化到 [0,1]²，同一个点可以映射到任意尺度；
② 偏移 `Δp` 与权重 `A` **都由 query 单独线性预测**（不做 Q·K 点积）；
③ `A` 对 **L×K 个点整体** softmax，所以跨尺度是竞争关系。

In [ ]:
LEVELS = [(100, 167), (50, 84), (25, 42), (13, 21)]   # 1333x800 输入的 stride 8/16/32/64
CH = 16
feats = [rng.normal(size=(h, w, CH)) for h, w in LEVELS]
L, K, NQ = len(LEVELS), 4, 6

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def ms_deform_attn(ref_pts, offsets, attn_logits, feats):
    '''ref_pts:(Nq,2) 归一化 (x,y) in [0,1]; offsets:(Nq,L,K,2) 归一化偏移;
       attn_logits:(Nq,L*K)。返回 out:(Nq,C), locs:(Nq,L,K,2), vals:(Nq,L*K,C)'''
    Nq = ref_pts.shape[0]; Kk = offsets.shape[2]; Cc = feats[0].shape[2]
    A = softmax(attn_logits, axis=-1)                    # 对 L*K 个点整体归一化
    out = np.zeros((Nq, Cc)); locs = []; vals = []
    for l, f in enumerate(feats):
        h, w, _ = f.shape
        p = ref_pts[:, None, :] + offsets[:, l, :, :]    # (Nq,K,2) 仍是归一化坐标
        pix = p * np.array([w - 1.0, h - 1.0])           # -> 该层的像素坐标
        v = bilinear_sample(f, pix.reshape(-1, 2)).reshape(Nq, Kk, Cc)
        out += (A[:, l*Kk:(l+1)*Kk][:, :, None] * v).sum(axis=1)
        locs.append(p); vals.append(v)
    return out, np.stack(locs, 1), np.concatenate(vals, axis=1), A

ref = rng.uniform(0.2, 0.8, size=(NQ, 2))
off = rng.normal(scale=0.03, size=(NQ, L, K, 2))         # 初始化时偏移很小
logits = rng.normal(scale=0.1, size=(NQ, L*K))
out, locs, vals, A = ms_deform_attn(ref, off, logits, feats)

print('输出 shape', out.shape, '| 采样点总数/query =', L*K)
print('注意力权重每行之和 =', np.round(A.sum(1), 8))
assert np.allclose(A.sum(1), 1.0)
# 输出是采样值的凸组合 -> 必须落在采样值的逐通道 min/max 之间
assert (out <= vals.max(1) + 1e-9).all() and (out >= vals.min(1) - 1e-9).all()
print()
print('query 0 的 16 个采样点（归一化坐标，按尺度分组）:')
for l in range(L):
    print('  level %d (stride %2d): ' % (l, 8 << l),
          np.round(locs[0, l], 3).tolist())
print()
print('✅ 同一个参考点被映射到 4 个尺度 -> **一套 query 同时看多尺度**，这是 DETR 做不到的。')

In [ ]:
# 计算量账：为什么 DETR 做不了多尺度，而可变形注意力可以
Cdim, Nq_real = 256, 300
hw_per_level = [(800 // s) * (1333 // s) for s in (8, 16, 32, 64)]
hw_c5 = hw_per_level[2]
hw_all = sum(hw_per_level)

detr_single = 2 * hw_c5 * Cdim                     # QK^T + AV，每 query
detr_multi  = 2 * hw_all * Cdim
deform      = L*K*Cdim + Cdim*(2*L*K) + Cdim*(L*K)  # 采样加权 + 预测偏移 + 预测权重

rows = [('DETR 单尺度 C5 (HW=%d)' % hw_c5, detr_single),
        ('DETR 若做 4 尺度 (HW=%d)' % hw_all, detr_multi),
        ('Deformable 4 尺度 K=4', deform)]
print('%-32s %16s %16s %8s' % ('方案', '每 query 乘加', '总量(x300)', '相对'))
for name, per_q in rows:
    print('%-32s %16.2e %16.2e %7.3fx' % (name, per_q, per_q*Nq_real, per_q/detr_single))
assert detr_multi / detr_single > 20, '多尺度全局注意力比单尺度贵 20 倍以上'
assert deform < detr_single / 20, '可变形注意力比单尺度全局注意力便宜一个数量级以上'
print()
print('⚠️  原版 DETR 不是「忘了」做多尺度 —— 是全局注意力在多尺度上算不起。')
print('    （encoder 的 self-attention 更是 O((HW)^2)，4 尺度下 %.1e 次乘加，直接爆炸）'
      % (hw_all**2 * Cdim))
print('✅ 把复杂度从「正比于特征图面积」改成「正比于采样点数」，多尺度才第一次变得免费。')

## 3 · 根因①：初始注意力质量 α，以及「注意力要多久才变尖锐」

训练刚开始时 Q/K 投影接近 0，softmax 近似均匀 -> 每个空间位置分到 1/HW 的权重。
**落在 GT 框内的注意力质量 α₀ = 目标占的格子数 / HW。**
这一节把 α₀ 算出来，并用一个玩具模型估计「α 要涨到 50% 需要多少 epoch」。

In [ ]:
def alpha0(box_px, stride, img_hw=(800, 1333)):
    '''初始（均匀）注意力落在 GT 框内的质量。'''
    Hf, Wf = img_hw[0] // stride, img_hw[1] // stride
    cells = (box_px[0] / stride) * (box_px[1] / stride)
    return cells, Hf * Wf, cells / (Hf * Wf)

CASES = [('大目标 近处车辆', (256, 256)), ('中目标 行人', (64, 128)),
         ('COCO 平均目标', (100, 100)),
         ('TSR 30m 限速牌', (32, 32)), ('TSR 60m 限速牌', (16, 16))]
print('%-20s %10s %12s %14s %12s' % ('场景 (stride 32)', '像素', '占格子数', 'HW', 'alpha0'))
alphas = {}
for name, wh in CASES:
    cells, hw, a = alpha0(wh, 32)
    alphas[name] = a
    print('%-20s %10s %12.3f %14d %11.4f%%' % (name, '%dx%d' % wh, cells, hw, a*100))
assert alphas['TSR 60m 限速牌'] < alphas['大目标 近处车辆'] / 200
print()
print('⚠️  60 米外的限速牌：**99.976%% 的 cross-attention 梯度打在背景上**。')
print('    这就是「DETR 收敛慢」在小目标上被放大的那一半原因。')

In [ ]:
# 玩具模型：注意力对数几率随训练线性增长，问「alpha 涨到 50% 要多少 epoch」
def mass_at_sharpness(n_in, n_tot, s):
    '''框内格点的 logit 抬高 s 之后的注意力质量。'''
    a = n_in * math.exp(s)
    return a / (a + (n_tot - n_in))

def epochs_to_half(n_in, n_tot, rate=0.02):
    '''logit 每 epoch 抬高 rate；解 mass=0.5 -> exp(s) = (n_tot-n_in)/n_in。'''
    return math.log((n_tot - n_in) / n_in) / rate

print('%-20s %10s %14s %16s' % ('场景', '占格子', 'alpha0', 'alpha->50% 需要 epoch'))
ep = {}
for name, wh in CASES:
    cells, hw, a = alpha0(wh, 32)
    e = epochs_to_half(cells, hw)
    ep[name] = e
    print('%-20s %10.3f %13.4f%% %16.0f' % (name, cells, a*100, e))
assert ep['TSR 60m 限速牌'] > ep['大目标 近处车辆'], '小目标需要更多 epoch 才能把注意力拧过去'
print()
print('（这是一个刻意简化的玩具模型，只用来说明「目标越小，注意力越难聚焦」的**量级关系**）')
print('✅ 它复现了两件事：① 500 epoch 这个量级 ② 小目标比大目标慢得多。')
print()
# 可变形注意力：参考点在框中心，采样点落在框内的比例就是它的 alpha0
def deform_alpha(box_px, mode, k=4, fixed_px=16.0, trials=4000, seed=1):
    '''mode="box-scaled": 偏移 ~ N(0,(0.25w)^2)，**按框宽高缩放**（Deformable/DINO 的做法）
       mode="fixed-px" : 偏移 ~ N(0,fixed_px^2)，与框大小无关（错误做法）'''
    r = np.random.default_rng(seed)
    w, h = box_px
    if mode == 'box-scaled':
        o = r.normal(size=(trials, k, 2)) * np.array([0.25*w, 0.25*h])
    else:
        o = r.normal(size=(trials, k, 2)) * fixed_px
    inside = (np.abs(o[..., 0]) <= w/2) & (np.abs(o[..., 1]) <= h/2)
    return inside.mean()

print()
print('%-20s %14s %18s %18s' % ('场景', '全局 alpha0', '可变形(按框缩放)', '可变形(固定±16px)'))
for name, wh in CASES:
    print('%-20s %13.4f%% %17.1f%% %17.1f%%'
          % (name, alpha0(wh, 32)[2]*100,
             deform_alpha(wh, 'box-scaled')*100, deform_alpha(wh, 'fixed-px')*100))
assert deform_alpha((16, 16), 'box-scaled') > 0.85
assert deform_alpha((16, 16), 'fixed-px') < 0.25
assert deform_alpha((256, 256), 'fixed-px') > 0.95
print()
print('✅ 可变形注意力把 alpha0 从 0.02% 拉到 ~90% —— 它不是「学会去看」，')
print('   而是**一开始就只看参考点附近**。这是根因① 的完整解法。')
print('⚠️  但前提是**偏移必须按参考框的宽高缩放**：用固定像素尺度时，')
print('    16x16 的标志只有 %.0f%% 的采样点落在框内，而 256x256 的车辆是 %.0f%%。'
      % (deform_alpha((16, 16), 'fixed-px')*100, deform_alpha((256, 256), 'fixed-px')*100))
print('    这就是 DAB-DETR / DINO 把 query 显式解释为 4D 框、并用 (w,h) 调制偏移的原因。')

## 4 · 根因②：匹配翻转率 φ

先从零实现匈牙利算法（用暴力枚举对拍），再模拟训练噪声下**同一个 GT 被不同 query 认领**的频率。
**φ 高 = 优化目标在抖 = 梯度方向掉头。**

In [ ]:
def hungarian(cost):
    '''O(n^3) 匈牙利算法（JV 势函数版本），要求 行数 <= 列数。返回 (rows, cols)。'''
    a = np.asarray(cost, dtype=float)
    n, m = a.shape
    assert n <= m, '要求行数 <= 列数'
    INF = float('inf')
    u = np.zeros(n + 1); v = np.zeros(m + 1)
    p = np.zeros(m + 1, dtype=int); way = np.zeros(m + 1, dtype=int)
    for i in range(1, n + 1):
        p[0] = i; j0 = 0
        minv = np.full(m + 1, INF); used = np.zeros(m + 1, dtype=bool)
        while True:
            used[j0] = True
            i0 = p[j0]; delta = INF; j1 = 0
            for j in range(1, m + 1):
                if used[j]:
                    continue
                cur = a[i0 - 1, j - 1] - u[i0] - v[j]
                if cur < minv[j]:
                    minv[j] = cur; way[j] = j0
                if minv[j] < delta:
                    delta = minv[j]; j1 = j
            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta; v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while j0:
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
    rows = np.arange(n); cols = np.zeros(n, dtype=int)
    for j in range(1, m + 1):
        if p[j]:
            cols[p[j] - 1] = j - 1
    return rows, cols

# 与暴力枚举对拍（含矩形情形）
for _ in range(30):
    n_ = int(rng.integers(2, 6)); m_ = n_ + int(rng.integers(0, 3))
    Cm = rng.random((n_, m_))
    r, c = hungarian(Cm)
    best = min(sum(Cm[i, pm[i]] for i in range(n_))
               for pm in itertools.permutations(range(m_), n_))
    assert len(set(c.tolist())) == n_, '一对一约束'
    assert abs(Cm[r, c].sum() - best) < 1e-9, (Cm[r, c].sum(), best)
print('✅ 匈牙利算法与暴力枚举在 30 组随机矩阵（含矩形）上完全一致')

In [ ]:
def flip_rate(n_epochs, Nq, M, noise, rng, base=None):
    '''同一张图连续 n_epochs 次匹配，统计「GT 换了认领 query」的比例。'''
    base = rng.random((M, Nq)) if base is None else base
    prev = None; flips = []
    for _ in range(n_epochs):
        cost = base + noise * rng.normal(size=(M, Nq))
        _, cols = hungarian(cost)
        if prev is not None:
            flips.append(float((cols != prev).mean()))
        prev = cols
    return float(np.mean(flips))

Nq, M = 100, 8
base = rng.random((M, Nq))
print('%-16s %14s' % ('代价噪声 sigma', '匹配翻转率 phi'))
res = {}
for s in [0.0, 0.02, 0.05, 0.1, 0.2, 0.4]:
    f = flip_rate(30, Nq, M, s, np.random.default_rng(7), base=base)
    res[s] = f
    print('%-16.2f %13.1f%%' % (s, f*100))
assert res[0.0] == 0.0, '无噪声时匹配完全稳定'
assert res[0.4] > res[0.05] > 0, '噪声越大翻转越多'
print()
print('⚠️  训练早期，分类分数几乎随机、框预测也差 -> 代价矩阵基本被噪声主导 -> phi 接近随机重排。')
print('    被翻转的那个 query 上个 epoch 在学「停车让行」，这个 epoch 目标变成了背景。')
print('    **梯度不是变小，是掉头。**')
print('✅ 这与「学习率太大」的症状完全不同：loss 曲线看着在降，但每个 query 学到的东西被反复覆盖。')

In [ ]:
# 训练推进 = 代价矩阵的「信号」变强（真实匹配越来越明显）-> phi 自然下降
print('%-14s %14s %16s' % ('训练阶段', '信噪比 (signal/noise)', '匹配翻转率 phi'))
stages = [('第 1 epoch', 0.3), ('第 10 epoch', 1.0), ('第 50 epoch', 3.0), ('第 200 epoch', 10.0)]
prev_f = 1.0
for name, snr in stages:
    f = flip_rate(30, Nq, M, 1.0/snr, np.random.default_rng(11), base=base)
    print('%-14s %20.1f %15.1f%%' % (name, snr, f*100))
    assert f <= prev_f + 1e-9
    prev_f = f
print()
print('这就是 DETR 的「自锁」：phi 高 -> 学不动 -> 信噪比涨不上去 -> phi 还是高。')
print('✅ DN-DETR 的做法是**绕开它**：额外塞一批监督目标已知、根本不经过匹配的 query，')
print('   让模型从第一个 iteration 就有一份 phi=0 的干净监督。')

## 5 · DN 去噪 query 的构造：噪声框 + 标签翻转

`box_noise_scale` λ：中心平移 ≤ λ/2·w，尺寸缩放 ∈ [1−λ, 1+λ]；
`label_noise_ratio` γ：以 γ 概率把标签换成**另一个**类。
这些 query 的监督目标是已知的 GT，**完全不经过匈牙利匹配**。

In [ ]:
def cxcywh_to_xyxy(b):
    cx, cy, w, h = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    return np.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], axis=1)

def iou_pairwise(a, b):
    '''a,b: (N,4) cxcywh，逐行配对求 IoU。'''
    A, B = cxcywh_to_xyxy(a), cxcywh_to_xyxy(b)
    x1 = np.maximum(A[:, 0], B[:, 0]); y1 = np.maximum(A[:, 1], B[:, 1])
    x2 = np.minimum(A[:, 2], B[:, 2]); y2 = np.minimum(A[:, 3], B[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    sa = (A[:, 2]-A[:, 0]) * (A[:, 3]-A[:, 1])
    sb = (B[:, 2]-B[:, 0]) * (B[:, 3]-B[:, 1])
    return inter / (sa + sb - inter + 1e-12)

def add_box_noise(boxes, lam, rng, lo=0.0):
    '''中心平移幅度 ∈ [lo/2, lam/2]·wh（随机符号），尺寸缩放 ∈ [1-lam, 1+lam]。'''
    cx, cy, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    n = len(boxes)
    mag_x = rng.uniform(lo/2, lam/2, n) * np.where(rng.random(n) < 0.5, -1.0, 1.0)
    mag_y = rng.uniform(lo/2, lam/2, n) * np.where(rng.random(n) < 0.5, -1.0, 1.0)
    sw = np.clip(1 + (rng.random(n)*2 - 1) * lam, 0.15, 3.0)   # 防止 lam>1 时宽高变负
    sh = np.clip(1 + (rng.random(n)*2 - 1) * lam, 0.15, 3.0)
    return np.stack([cx + mag_x*w, cy + mag_y*h, w*sw, h*sh], axis=1)

def flip_labels(labels, num_classes, ratio, rng):
    m = rng.random(len(labels)) < ratio
    out = labels.copy()
    step = rng.integers(1, num_classes, size=len(labels))   # 1..C-1 -> 保证换成**别的**类
    out[m] = (labels[m] + step[m]) % num_classes
    return out, m

NUM_CLASSES = 45                        # TSR 常见量级：几十个标志细类
M_GT = 6
gt_boxes = np.stack([rng.uniform(0.25, 0.75, M_GT), rng.uniform(0.25, 0.75, M_GT),
                     rng.uniform(0.04, 0.20, M_GT), rng.uniform(0.04, 0.20, M_GT)], axis=1)
gt_labels = rng.integers(0, NUM_CLASSES, M_GT)

print('%-8s %14s %14s' % ('lambda', '带噪框与GT的min IoU', 'mean IoU'))
for lam in [0.1, 0.2, 0.4, 0.8, 1.2]:
    nb = add_box_noise(np.repeat(gt_boxes, 200, axis=0), lam, np.random.default_rng(3))
    io = iou_pairwise(nb, np.repeat(gt_boxes, 200, axis=0))
    print('%-8.1f %18.3f %14.3f' % (lam, io.min(), io.mean()))
nb04 = add_box_noise(np.repeat(gt_boxes, 500, axis=0), 0.4, np.random.default_rng(3))
assert iou_pairwise(nb04, np.repeat(gt_boxes, 500, axis=0)).min() > 0.3, 'lam=0.4 的带噪框仍与 GT 大幅重叠'
print()
print('✅ lambda=0.4（DN-DETR 默认）时 IoU 下界约 0.36 —— 「差不多但不准」，正好是去噪任务想要的。')
print('⚠️  lambda 太大（>0.8）时带噪框与 GT 已不是同一个目标，去噪变成「无中生有」，制造错误监督。')

In [ ]:
# 标签翻转 + 完整的 DN 组构造
GAMMA, GROUPS = 0.3, 50

def build_dn_queries(gt_boxes, gt_labels, num_classes, groups, lam, gamma, rng):
    dnb, dnl, tb, tl, gid = [], [], [], [], []
    for g in range(groups):
        dnb.append(add_box_noise(gt_boxes, lam, rng))
        nl, _ = flip_labels(gt_labels, num_classes, gamma, rng)
        dnl.append(nl); tb.append(gt_boxes); tl.append(gt_labels)
        gid.append(np.full(len(gt_boxes), g))
    return (np.concatenate(dnb), np.concatenate(dnl), np.concatenate(tb),
            np.concatenate(tl), np.concatenate(gid))

dn_b, dn_l, tgt_b, tgt_l, gid = build_dn_queries(
    gt_boxes, gt_labels, NUM_CLASSES, GROUPS, 0.4, GAMMA, np.random.default_rng(5))
flipped = (dn_l != tgt_l)
print('GT 数 %d  x  去噪组数 %d  =  %d 个去噪 query' % (M_GT, GROUPS, len(dn_b)))
print('实际标签翻转率 %.3f （设定 gamma=%.2f）' % (flipped.mean(), GAMMA))
print('翻转后的标签都与原标签不同: ', bool((dn_l[flipped] != tgt_l[flipped]).all()))
assert abs(flipped.mean() - GAMMA) < 0.06
assert (dn_l[flipped] != tgt_l[flipped]).all()
assert len(dn_b) == M_GT * GROUPS
print()
print('监督密度对比（单张图、单次迭代）：')
print('  匹配分支   : %2d 个正样本，且认领关系每个 epoch 都可能翻转 (phi>0)' % M_GT)
print('  去噪分支   : %d 个正样本，监督目标已知、**phi = 0**' % len(dn_b))
print('  密度提升   : %.0f 倍' % (len(dn_b)/M_GT))
print('✅ 这就是 DN 把 50 epoch 压到 12 epoch 的全部秘密：**又多又稳的监督**。')

## 6 · attention mask 隔离：这个机制的成败全在这里

去噪 query 携带 GT 信息。**匹配 query 若能读到它们 = 训练时抄答案**。
注意官方实现里 mask **不是对称的**：去噪 query 可以看匹配 query，反过来不行。

In [ ]:
def dn_attention_mask(n_per_group, n_groups, n_match):
    '''True = 屏蔽。布局：[去噪 query (n_groups x n_per_group)] + [匹配 query (n_match)]'''
    n_dn = n_per_group * n_groups
    T = n_dn + n_match
    mask = np.zeros((T, T), dtype=bool)
    mask[n_dn:, :n_dn] = True                      # ① 匹配 query 看不见任何去噪 query
    g = np.repeat(np.arange(n_groups), n_per_group)
    mask[:n_dn, :n_dn] = g[:, None] != g[None, :]  # ② 去噪组之间互相看不见
    return mask                                    # ③ 去噪 -> 匹配 的列**不屏蔽**（非对称！）

n_per, n_g, n_m = 3, 3, 5
mk = dn_attention_mask(n_per, n_g, n_m)
n_dn = n_per * n_g
print('mask (True=屏蔽)，前 %d 行/列是去噪 query，后 %d 是匹配 query:' % (n_dn, n_m))
print('     ' + ' '.join('%2d' % j for j in range(mk.shape[1])))
for i, row in enumerate(mk):
    tag = 'DN%d' % (i // n_per) if i < n_dn else 'MAT'
    print('%-4s ' % tag + ' '.join(' X' if x else ' .' for x in row))

assert mk[n_dn:, :n_dn].all(), '① 匹配 query 必须完全看不见去噪 query（否则 GT 泄漏）'
gg = np.repeat(np.arange(n_g), n_per)
assert (mk[:n_dn, :n_dn] == (gg[:, None] != gg[None, :])).all(), '② 组间屏蔽、组内可见'
assert not mk[:n_dn, n_dn:].any(), '③ 去噪 query 可以看匹配 query —— **mask 是非对称的**'
assert not mk[n_dn:, n_dn:].any(), '匹配 query 之间照常做去重协商'
print()
print('✅ 三条规则全部成立。')
print('⚠️  忘记 ① 的症状极具欺骗性：训练 loss 掉得非常漂亮、去噪重建误差趋近 0，')
print('    但**验证 AP 比不加 DN 还差**。排查法：关掉 DN 重训一次，若 AP 反而涨了，八成是 mask 写错。')

In [ ]:
# 量化「泄漏」：如果 mask 写错（或忘了写），有多少条 GT 信息通道被打开
def leak_channels(mask, n_dn, n_match):
    '''匹配 query 能读到去噪 query 的 (query, dn) 对数。'''
    return int((~mask[n_dn:, :n_dn]).sum())

no_mask = np.zeros_like(mk)
sym_mask = mk | mk.T                    # 常见错误：想当然写成对称
print('%-28s %14s %22s' % ('mask 写法', '泄漏通道数', '去噪能否看到匹配上下文'))
for name, m_ in [('① 完全不加 mask', no_mask),
                 ('② 写成对称矩阵（常见错误）', sym_mask),
                 ('③ 官方写法（非对称）', mk)]:
    leak = leak_channels(m_, n_dn, n_m)
    ctx = '否' if m_[:n_dn, n_dn:].any() else '是'
    print('%-28s %14d %22s' % (name, leak, ctx))
assert leak_channels(no_mask, n_dn, n_m) == n_dn * n_m
assert leak_channels(sym_mask, n_dn, n_m) == 0 and sym_mask[:n_dn, n_dn:].any()
assert leak_channels(mk, n_dn, n_m) == 0 and not mk[:n_dn, n_dn:].any()
print()
print('✅ ② 虽然不泄漏，但**削弱了去噪分支从匹配分支获得的上下文**——')
print('   而去噪分支学到的定位能力本来是要迁移给匹配分支的。所以官方选了非对称。')

## 7 · CDN 对比去噪：正负噪声对

DINO 的关键补充：**只有正样本时，模型只被教「往 GT 靠」，从没被教「离太远就判背景」**——
于是重复框多、置信度不校准。CDN 在 λ1 与 λ2 之间造一圈**硬负样本**，目标是 no-object。

In [ ]:
LAM1, LAM2 = 0.4, 1.2
NO_OBJECT = NUM_CLASSES          # 背景类的类别 id

def build_cdn_queries(gt_boxes, gt_labels, num_classes, groups, lam1, lam2, gamma, rng):
    '''每组 2M 个 query：前 M 正（噪声<lam1，目标=GT），后 M 负（噪声∈(lam1,lam2)，目标=背景）。'''
    qb, ql, tb, tl, is_pos, gid = [], [], [], [], [], []
    M = len(gt_boxes)
    for g in range(groups):
        pb = add_box_noise(gt_boxes, lam1, rng, lo=0.0)
        pl, _ = flip_labels(gt_labels, num_classes, gamma, rng)
        nb = add_box_noise(gt_boxes, lam2, rng, lo=lam1)
        nl, _ = flip_labels(gt_labels, num_classes, gamma, rng)
        qb += [pb, nb]; ql += [pl, nl]
        tb += [gt_boxes, gt_boxes]
        tl += [gt_labels, np.full(M, num_classes)]      # 负样本的目标 = no-object
        is_pos += [np.ones(M, bool), np.zeros(M, bool)]
        gid += [np.full(M, g), np.full(M, g)]
    return (np.concatenate(qb), np.concatenate(ql), np.concatenate(tb),
            np.concatenate(tl), np.concatenate(is_pos), np.concatenate(gid))

qb, ql, tb2, tl2, ispos, gid2 = build_cdn_queries(
    gt_boxes, gt_labels, NUM_CLASSES, 200, LAM1, LAM2, GAMMA, np.random.default_rng(9))
io = iou_pairwise(qb, tb2)
print('正样本 IoU  中位数 %.3f  均值 %.3f  最小 %.3f' % (np.median(io[ispos]), io[ispos].mean(), io[ispos].min()))
print('负样本 IoU  中位数 %.3f  均值 %.3f  最大 %.3f' % (np.median(io[~ispos]), io[~ispos].mean(), io[~ispos].max()))
print('负样本的监督目标全部是 no-object(id=%d): %s' % (NO_OBJECT, bool((tl2[~ispos] == NO_OBJECT).all())))
assert io[ispos].mean() > io[~ispos].mean() + 0.15
assert np.median(io[ispos]) > np.median(io[~ispos])
assert (tl2[~ispos] == NO_OBJECT).all() and (tl2[ispos] == np.tile(gt_labels, 200)).all()
print()
print('✅ 正负两组的噪声尺度之差，**直接定义了「多近算认领成功」这条决策边界**。')
print('   这正是 NMS 在推理期做的事，被搬到了训练期 —— 所以 DINO 的重复框显著少于 DN-DETR。')
print('⚠️  注意负样本不是「完全不重叠」，而是「近但不够近」——太远的负样本是简单负样本，没有信息量。')

## 8 · look forward once vs twice：一次 detach 的代价

Deformable DETR 在层间对参考框 `detach()`（LFO）：第 i 层参数只被第 i 层的损失监督。
DINO 去掉这个 detach（LFT）：第 i 层参数同时被第 i、i+1 层的损失监督。

In [ ]:
def refine_chain(theta, b0, y, detach):
    '''3 层框细化：b_i = b_{i-1} + theta_i；每层都有辅助损失 (b_i - y)^2。
       返回 (总损失, dL/dtheta)。detach=True 时切断层间梯度（look forward once）。'''
    b = [b0]
    for t in theta:
        b.append(b[-1] + t)
    loss = sum((bi - y)**2 for bi in b[1:])
    g = np.zeros(len(theta))
    for i in range(len(theta)):
        if detach:
            g[i] = 2 * (b[i+1] - y)                       # 只有第 i 层自己的损失
        else:
            g[i] = sum(2 * (b[j+1] - y) for j in range(i, len(theta)))
    return loss, g

def curve(detach, steps=40, lr=0.02):
    theta = np.zeros(3); b0, y = 0.0, 1.0
    hist = []
    for _ in range(steps):
        loss, g = refine_chain(theta, b0, y, detach)
        hist.append(loss)
        theta -= lr * g
    return np.array(hist), theta

h_lfo, t_lfo = curve(True)
h_lft, t_lft = curve(False)
print('%-26s %9s %9s %9s %9s' % ('总损失', 'step 1', 'step 5', 'step 10', 'step 20'))
print('%-26s %9.4f %9.4f %9.4f %9.4f'
      % ('look forward once (detach)', h_lfo[0], h_lfo[4], h_lfo[9], h_lfo[19]))
print('%-26s %9.4f %9.4f %9.4f %9.4f'
      % ('look forward twice (DINO)', h_lft[0], h_lft[4], h_lft[9], h_lft[19]))
_, g_lfo = refine_chain(np.zeros(3), 0.0, 1.0, True)
_, g_lft = refine_chain(np.zeros(3), 0.0, 1.0, False)
print()
print('初始梯度对比 dL/dtheta:')
print('  LFO %s   <- 第 1 层只从 loss1 拿梯度' % np.round(g_lfo, 3))
print('  LFT %s   <- 第 1 层从 loss1+loss2+loss3 拿梯度' % np.round(g_lft, 3))
assert abs(g_lft[0]) > abs(g_lfo[0]), '第 1 层在 LFT 下拿到更多监督'
assert g_lft[-1] == g_lfo[-1], '最后一层两者相同'
assert h_lft[9] < h_lfo[9] and h_lft[19] < h_lfo[19], '同样步数下 LFT 下降更快'
print()
print('✅ 一次 detach 的代价：**早期层的框预测不为最终结果负责**。')
print('⚠️  代价也要说清楚：LFT 让早期层拿到 L 倍的梯度，等于给它们加了更大的有效学习率。')
print('    把 lr 从 0.02 提到 0.05 再跑一遍，LFT 反而会震荡到比 LFO 更差 —— ')
print('    **换 LFT 时学习率要相应保守**，这在真实训练里同样成立。')
print('⚠️  而且顺序不能颠倒 —— Deformable DETR 当初 detach 是因为早期框太差、梯度互相干扰；')
print('    是 DN 先让早期层的框变准了，去掉 detach 才划算。「前一个改进让后一个可行」。')

## 9 · 一对多监督密度的账：反推 Group DETR 的加速倍数

**一对一是推理端的需求，不是训练端的最优。**
这一节把「每图每次迭代的正样本数」算清楚，并反推收敛 epoch。

In [ ]:
COCO_IMGS, GT_PER_IMG = 118287, 7.3
METHODS = [
    ('Faster R-CNN (采样后)', 256.0, '有 -> 需 NMS'),
    ('RetinaNet / FCOS (密集)', 600.0, '有 -> 需 NMS'),
    ('DETR (一对一)', GT_PER_IMG, '基本没有'),
    ('Group DETR (G=11)', GT_PER_IMG * 11, '推理只留 1 组'),
    ('H-DETR (一对多 K=6)', GT_PER_IMG * 7, '推理丢一对多分支'),
    ('Co-DETR (+密集辅助头)', GT_PER_IMG * 40, '推理丢辅助头'),
]
BUDGET = COCO_IMGS * GT_PER_IMG * 500          # DETR 训 500 epoch 累计的正样本梯度更新数
print('%-26s %14s %10s %12s %s' % ('检测器', '正样本/图/迭代', '相对DETR', '等效epoch', '推理端重复框'))
eps = {}
for name, npos, dup in METHODS:
    e = BUDGET / (COCO_IMGS * npos)
    eps[name] = e
    print('%-26s %14.1f %9.1fx %11.0f  %s' % (name, npos, npos/GT_PER_IMG, e, dup))
assert eps['DETR (一对一)'] == 500
assert 40 < eps['Group DETR (G=11)'] < 50, 'Group DETR 论文报告的加速正是这个量级'
assert eps['Co-DETR (+密集辅助头)'] < eps['H-DETR (一对多 K=6)'] < eps['DETR (一对一)']
print()
print('✅ 「500 / 11 ≈ 45 epoch」—— 与 Group DETR 论文报告的收敛加速惊人地一致。')
print('✅ 密集检测器每次迭代从一张图榨出几百条监督，DETR 只榨 7.3 条。')
print('   要拿到同样多的梯度更新，DETR 自然需要几十倍 epoch —— 这是根因的第三个角度。')

In [ ]:
# 一对多分支必须与一对一分支**分开做 self-attention**，否则去重信号被搅乱
def dedup_signal(n_o2o, n_o2m, shared):
    '''粗糙模型：self-attention 里若混入 n_o2m 个「本来就该重复」的 query，
       去重信号被稀释的比例。'''
    if not shared:
        return 1.0
    return n_o2o / (n_o2o + n_o2m)

print('%-34s %16s' % ('配置', '一对一分支的去重信号强度'))
for name, shared in [('分开 self-attention（Group/H-DETR 的做法）', False),
                     ('混在同一次 self-attention（错误做法）', True)]:
    print('%-34s %15.1f%%' % (name, dedup_signal(300, 1500, shared)*100))
assert dedup_signal(300, 1500, True) < 0.25 < dedup_signal(300, 1500, False)
print()
print('⚠️  混着做的后果：**一对一分支也开始输出重复框，NMS-free 的性质直接丢失。**')
print('    这与 DN 的 attention mask 是同一类设计约束 —— 训练期加进来的额外 query，')
print('    必须用 mask 把它们与主分支的协商过程隔开。')
print()
print('工程结论：如果推理端仍然要加 NMS，就没必要坚持一对一匹配。')
print('  「训练慢了几倍、部署还是要 NMS」是最糟糕的组合。')

## ✏️ 练习 1：按参考框缩放采样偏移

实现 `scale_offsets_by_box(ref_boxes, raw_offsets, factor=0.5)`：
`ref_boxes` 是 `(Nq,4)` 的 cxcywh 归一化框，`raw_offsets` 是 `(Nq,L,K,2)` 的原始预测。
返回 `(sampling_locations, scaled_offsets)`——**偏移要乘以 `factor * (w,h)`**，
采样点 = 框中心 + 缩放后的偏移。这是 DAB-DETR / DINO 让采样范围随目标尺度自适应的做法。

In [ ]:
def scale_offsets_by_box(ref_boxes, raw_offsets, factor=0.5):
    # TODO: ① 从 ref_boxes 取出中心 (cx,cy) 与宽高 (w,h)
    #       ② scaled = raw_offsets * factor * (w,h)，注意广播到 (Nq,L,K,2)
    #       ③ locations = (cx,cy) + scaled
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rb = np.array([[0.5, 0.5, 0.02, 0.02],      # 小框（约 27x16 px）
               [0.5, 0.5, 0.40, 0.40]])     # 大框
raw = np.ones((2, 2, 3, 2))                 # 全 1 偏移，方便手算
loc, sc = scale_offsets_by_box(rb, raw, factor=0.5)
assert loc.shape == (2, 2, 3, 2) and sc.shape == (2, 2, 3, 2)
assert np.allclose(sc[0], 0.5*0.02), '小框：偏移应被缩放到 0.01'
assert np.allclose(sc[1], 0.5*0.40), '大框：偏移应被缩放到 0.20'
assert np.allclose(loc[0], 0.5 + 0.01) and np.allclose(loc[1], 0.5 + 0.20)
raw2 = np.repeat(np.random.default_rng(0).normal(size=(1, 2, 3, 2)), 2, axis=0)  # 两行共用同一组原始偏移
loc2, sc2 = scale_offsets_by_box(rb, raw2)
spread = np.abs(sc2).mean(axis=(1, 2, 3))
print('小框采样点平均散布 %.4f  |  大框 %.4f  |  比值 %.1f'
      % (spread[0], spread[1], spread[1]/spread[0]))
assert abs(spread[1]/spread[0] - 20.0) < 1e-6, '散布之比应等于宽高之比 0.40/0.02 = 20'
print('✅ 练习 1 通过：**采样范围随目标尺度自适应** —— 小目标不会把点撒到框外去。')

## ✏️ 练习 2：DN 的 attention mask

实现 `build_dn_attn_mask(n_per_group, n_groups, n_match)`，返回 `(T,T)` 的布尔矩阵，
`True = 屏蔽`。布局是 `[去噪 query] + [匹配 query]`。三条规则：
① 匹配 query 看不见任何去噪 query；② 去噪组之间互不可见（组内可见）；
③ **去噪 query 可以看匹配 query（非对称！）**。

In [ ]:
def build_dn_attn_mask(n_per_group, n_groups, n_match):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
for npg, ng, nm in [(3, 3, 5), (2, 5, 10), (1, 1, 4), (4, 2, 1)]:
    mm = build_dn_attn_mask(npg, ng, nm)
    nd = npg * ng
    assert mm.shape == (nd + nm, nd + nm) and mm.dtype == bool
    assert mm[nd:, :nd].all(), '① 匹配 query 必须看不见去噪 query'
    gg_ = np.repeat(np.arange(ng), npg)
    assert (mm[:nd, :nd] == (gg_[:, None] != gg_[None, :])).all(), '② 组间屏蔽、组内可见'
    assert not mm[:nd, nd:].any(), '③ 去噪 query 可以看匹配 query'
    assert not mm[nd:, nd:].any(), '匹配 query 之间照常协商'
mm = build_dn_attn_mask(2, 3, 4)
print('屏蔽率 %.1f%%（%d/%d 个注意力对被切断）'
      % (mm.mean()*100, mm.sum(), mm.size))
assert not (mm == mm.T).all(), 'mask 必须是**非对称**的'
print('✅ 练习 2 通过：mask 写错 = 训练时抄答案，是 DN 最容易踩的坑。')

## ✏️ 练习 3：CDN 的监督目标

实现 `cdn_targets(query_boxes, gt_boxes, gt_labels, is_positive, num_classes)`：
返回 `(target_labels, target_boxes, box_loss_mask)`。
规则：正样本 → 目标类别是 GT 类别、要算框损失；
**负样本 → 目标类别是 `num_classes`（no-object）、不算框损失**（背景没有框可回归）。

In [ ]:
def cdn_targets(query_boxes, gt_boxes, gt_labels, is_positive, num_classes):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
gb = np.array([[0.3, 0.3, 0.1, 0.1], [0.7, 0.6, 0.2, 0.15]])
gl = np.array([4, 11])
qb_ = np.concatenate([gb + 0.01, gb + 0.15])
gtl = np.concatenate([gb, gb])
gll = np.concatenate([gl, gl])
pos = np.array([True, True, False, False])
tl_, tb_, bm_ = cdn_targets(qb_, gtl, gll, pos, num_classes=45)
assert tl_.tolist() == [4, 11, 45, 45], tl_.tolist()
assert np.allclose(tb_[pos], gb) and bm_.tolist() == [True, True, False, False]
assert (tl_[~pos] == 45).all(), '负样本必须是 no-object'
print('target labels:', tl_.tolist())
print('参与框损失的 query 数: %d / %d' % (bm_.sum(), len(bm_)))
print('✅ 练习 3 通过：**负样本只贡献分类损失** —— 忘了 mask 掉框损失会把背景框拉向 GT，')
print('   等于把 CDN 的负样本又变回了正样本。')

## ✏️ 练习 4：监督密度台账

实现 `supervision_density(n_pos, alpha, phi)`：返回 `rho = n_pos * alpha * (1-phi)`，
以及 `rank_methods(table)`：给定 `{名字: (n_pos, alpha, phi)}`，
按 rho 从小到大返回名字列表。用它复现 DETR → Deformable → DN → DINO → Co-DETR 的演进顺序。

In [ ]:
def supervision_density(n_pos, alpha, phi):
    # TODO
    raise NotImplementedError

def rank_methods(table):
    # TODO: 返回按 rho 升序排列的名字列表
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
LEDGER = {                       # (每图每次迭代正样本数, 注意力落在目标上的质量, 匹配翻转率)
    'DETR':            (7.3,   0.0010, 0.50),
    'Deformable DETR': (7.3,   0.9000, 0.45),
    'DN-DETR':         (7.3 + 5*7.3,  0.9000, 0.10),
    'DINO':            (7.3 + 100*2*7.3, 0.9200, 0.08),
    'Co-DETR':         (7.3 + 100*2*7.3 + 300.0, 0.9200, 0.08),
}
assert abs(supervision_density(10, 0.5, 0.2) - 4.0) < 1e-9
order = rank_methods(LEDGER)
print('%-18s %12s %10s %8s %14s' % ('方法', 'n_pos', 'alpha', 'phi', 'rho'))
for k in order:
    n, a, p = LEDGER[k]
    print('%-18s %12.1f %10.4f %8.2f %14.2f' % (k, n, a, p, supervision_density(n, a, p)))
assert order == ['DETR', 'Deformable DETR', 'DN-DETR', 'DINO', 'Co-DETR'], order
assert supervision_density(*LEDGER['Deformable DETR']) / supervision_density(*LEDGER['DETR']) > 500
print()
print('✅ 练习 4 通过：**一条主线** —— 每一步演进都在抬高 rho 的某一个因子。')
print('   面试里把这张台账口述出来，比背模型列表有效得多。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def scale_offsets_by_box(ref_boxes, raw_offsets, factor=0.5):
    ctr = ref_boxes[:, None, None, :2]                 # (Nq,1,1,2)
    wh = ref_boxes[:, None, None, 2:]                  # (Nq,1,1,2)
    scaled = raw_offsets * factor * wh
    return ctr + scaled, scaled

In [ ]:
# 练习 2 参考答案
def build_dn_attn_mask(n_per_group, n_groups, n_match):
    n_dn = n_per_group * n_groups
    T = n_dn + n_match
    mask = np.zeros((T, T), dtype=bool)
    mask[n_dn:, :n_dn] = True                                  # ① 匹配看不见去噪
    g = np.repeat(np.arange(n_groups), n_per_group)
    mask[:n_dn, :n_dn] = g[:, None] != g[None, :]              # ② 组间屏蔽
    return mask                                                # ③ 去噪->匹配 不屏蔽

In [ ]:
# 练习 3 参考答案
def cdn_targets(query_boxes, gt_boxes, gt_labels, is_positive, num_classes):
    is_positive = np.asarray(is_positive, dtype=bool)
    target_labels = np.where(is_positive, gt_labels, num_classes)
    target_boxes = gt_boxes.copy()
    box_loss_mask = is_positive.copy()                         # 负样本不算框损失
    return target_labels, target_boxes, box_loss_mask

In [ ]:
# 练习 4 参考答案
def supervision_density(n_pos, alpha, phi):
    return n_pos * alpha * (1.0 - phi)

def rank_methods(table):
    return sorted(table, key=lambda k: supervision_density(*table[k]))

---
## 🧪 真实工程胶囊：DINO 风格的训练配置与去噪模块骨架

In [ ]:
RECIPE = r'''
# ============ 1) mmdetection / detrex 风格的 DINO 配置要点 ============
model = dict(
    type='DINO',
    num_queries=900,                     # DINO 用 900（DETR 是 100）；密集场景要更多
    num_feature_levels=4,                # **多尺度是准入条件**；TSR 建议把 P2(stride4) 也评估进来
    with_box_refine=True,                # 迭代框细化
    as_two_stage=True,                   # encoder 出提案 -> top-K 作参考点
    dn_cfg=dict(                         # ---- 去噪（DN / CDN）----
        label_noise_scale=0.5,           # gamma：标签翻转比例
        box_noise_scale=1.0,             # lambda1：正样本噪声上界（DINO 用 1.0）
        group_cfg=dict(dynamic=True, num_groups=None, num_dn_queries=100),
    ),                                   # dynamic=True: GT 多的图自动减组数，防显存爆
    encoder=dict(num_layers=6, layer_cfg=dict(
        self_attn_cfg=dict(embed_dims=256, num_levels=4, dropout=0.0))),
    decoder=dict(num_layers=6, return_intermediate=True,   # 每层都算辅助损失
                 layer_cfg=dict(cross_attn_cfg=dict(num_levels=4, num_points=4))),
)

# ============ 2) 去噪 query 的构造（核心 20 行）============
def prepare_dn(gt_boxes, gt_labels, num_classes, num_groups,
               lam1=1.0, lam2=2.0, gamma=0.5):
    # 返回 (dn_query_boxes, dn_query_labels, targets, attn_mask)
    # 每组 2M 个 query：前 M 正（噪声<lam1），后 M 负（lam1<噪声<lam2）
    M = len(gt_boxes)
    known = gt_boxes.repeat(2 * num_groups, 1)            # 正负各一份 x G 组
    labels = gt_labels.repeat(2 * num_groups)
    # -- 标签翻转（只对正样本组或全部，看实现）--
    flip = torch.rand_like(labels.float()) < gamma
    labels[flip] = torch.randint_like(labels[flip], 0, num_classes)
    # -- 框噪声：中心平移 + 尺寸缩放，负样本用更大的噪声尺度 --
    neg = torch.zeros(2 * num_groups * M, dtype=torch.bool)
    neg[M::2 * M] = True                                  # 具体索引按 layout 定
    scale = torch.where(neg, lam2, lam1)
    delta = (torch.rand_like(known) * 2 - 1) * scale[:, None] * 0.5
    known[:, :2] += delta[:, :2] * known[:, 2:]           # **平移按框宽高缩放**
    known[:, 2:] *= 1 + delta[:, 2:]
    known = known.clamp(min=1e-4, max=1.0)
    return known, labels, neg

# ============ 3) attention mask：非对称，写错就是抄答案 ============
def dn_attn_mask(pad_size, num_groups, num_queries, single_pad):
    T = pad_size + num_queries
    m = torch.zeros(T, T, dtype=torch.bool)
    m[pad_size:, :pad_size] = True                        # 匹配 query 看不见去噪 query
    for i in range(num_groups):                           # 去噪组之间互不可见
        s, e = single_pad * 2 * i, single_pad * 2 * (i + 1)
        m[s:e, e:pad_size] = True
        m[s:e, :s] = True
    return m                                              # 去噪 -> 匹配 的列**不屏蔽**

# ============ 4) 上线前的三条自检 ============
# [1] 关掉 DN 重训一次：若验证 AP 反而更高 -> attention mask 写错了（GT 泄漏）
# [2] 打印去噪分支与匹配分支各自的 loss 曲线：去噪 loss 应在前 1000 iter 内快速下降
# [3] 按**像素尺寸分桶**看 AP：只看总 mAP 会掩盖「小目标层没接上」这类严重问题
'''
print(RECIPE)
for key in ['num_feature_levels=4', 'dn_cfg', 'box_noise_scale', 'attn_mask',
            'm[pad_size:, :pad_size] = True', '按框宽高缩放', '分桶']:
    assert key in RECIPE, key
print('✅ 配方覆盖：多尺度 / 去噪超参 / 动态组数 / 非对称 mask / 上线自检')

### 小结

- **「DETR 收敛慢」要拆成两个正交根因**：① cross-attention 初期近似均匀，梯度打在错误的**空间位置**；
  ② 二分匹配翻转，梯度指向错误的**目标**。修一个另一个还在，所以两次收益可以相乘。
- **根因① 在小目标上被放大**：60 米外 16×16 的限速牌在 stride 32 特征图上占 0.25 格，
  初始注意力质量只有 **0.024%**。→ **对 TSR，多尺度不是调参选项，是准入条件。**
- **可变形注意力的两个关键点**：① 权重由 query 单独产生，**不做 Q·K 点积**（所以便宜）；
  ② 双线性插值让**采样位置本身可微**，模型有一条「学会往哪看」的直达梯度通道。
  偏移必须**按参考框宽高缩放**，否则小目标的采样点全撒到框外。
- **DN 的本质是「绕开匹配」而不是「改进匹配」**：带噪 GT 作为额外 query，监督目标已知、
  φ=0、密度高几十倍，而且**推理零开销**（去噪 query 训练时才存在）。
- **attention mask 是 DN 的成败所在，而且是非对称的**：匹配 query 看不见去噪 query，
  去噪组之间互不可见，但**去噪 query 可以看匹配 query**。写错的症状是
  「train loss 极漂亮、val AP 反而更差」。
- **DINO 的三件事各修一个短板**：CDN 用负样本教决策边界（把 NMS 的职责搬到训练期）；
  mixed query selection **只借位置不借内容**；look forward twice 让早期层为最终结果负责。
- **一对一是推理端的需求，不是训练端的最优**——Group/H-/Co-DETR 把一对多加回训练，
  推理时整块丢掉。「500 epoch ÷ 11 组 ≈ 45 epoch」这笔账值得记住。
- **一句话主线**：ρ ∝ (被监督的 query 数) × (注意力落在目标上的质量) × (匹配未翻转比例)，
  **每一步演进都在抬高其中一个因子**。

下一站：**模块 05 · DETR 工程实践** —— 学习率分组、诊断树、以及给 TSR 到底该选 DETR 还是 YOLO。